In [4]:
# Задание task_03_04_07.
#Задание: На сайте Всемирного банка (WB) в разделе Data доступна экономическая статистика о валовом внутреннем продукте (ВВП) на душу населения в долларах США 1.
#Используя подготовленный CSV-файл, реализуйте: загрузку данных; поиск государства по названию, а также государства с максимальным, минимальным ВВП на душу населения; сохранение данных в новый CSV-файл с фильтром по определенному условию (например, топ-10 государств по объему ВВП на душу населения).
# Выполнила: Михалева Полина Вячеславовна
# Группа: ЦИБ-251

from google.colab import files
uploaded = files.upload()

import csv
import os
import unittest


class NoSuchCountryError(Exception):
    pass


class IllegalArgumentError(ValueError):
    pass


class DataLoadError(Exception):
    pass


def load_data(filename):
    if not isinstance(filename, str):
        raise TypeError("Имя файла должно быть строкой.")

    data = []
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            reader = csv.DictReader(file)
            if reader.fieldnames is None or 'name' not in reader.fieldnames or 'gdp' not in reader.fieldnames:
                raise DataLoadError("CSV-файл должен содержать заголовки 'name' и 'gdp'.")

            for row in reader:
                name = str(row.get('name', '')).strip()
                gdp_str = str(row.get('gdp', '')).strip()

                if not name or not gdp_str:
                    continue

                try:
                    gdp_value = float(gdp_str)
                    data.append({'name': name, 'gdp': gdp_value})
                except ValueError:
                    continue
    except FileNotFoundError:
        raise FileNotFoundError(f"Файл '{filename}' не найден.")
    except PermissionError:
        raise PermissionError(f"Нет прав на чтение файла '{filename}'.")
    except UnicodeDecodeError:
        raise DataLoadError(f"Ошибка кодировки файла '{filename}'. Ожидается UTF-8.")
    except DataLoadError:
        raise
    except Exception as e:
        raise DataLoadError(f"Неожиданная ошибка при чтении файла: {e}")

    return data


def search(data, criteria):
    if not isinstance(data, list):
        raise TypeError("Параметр 'data' должен быть списком словарей.")
    if not isinstance(criteria, str):
        raise TypeError("Параметр 'criteria' должен быть строкой.")

    if not data:
        raise NoSuchCountryError("Список данных пуст, поиск невозможен.")

    if criteria == "-max-":
        return max(data, key=lambda x: x['gdp'])
    elif criteria == "-min-":
        return min(data, key=lambda x: x['gdp'])
    else:
        for item in data:
            if item['name'].lower() == criteria.lower():
                return item
        raise NoSuchCountryError(f"Страна '{criteria}' не найдена в загруженных данных.")


def save_data(filename, data, criteria):
    if not isinstance(filename, str) or not filename.strip():
        raise TypeError("Имя файла для сохранения должно быть непустой строкой.")
    if not isinstance(data, list):
        raise TypeError("Параметр 'data' должен быть списком.")
    if not isinstance(criteria, str) or '=' not in criteria:
        raise IllegalArgumentError("Критерий должен быть строкой формата 'метод=значение' (например, 'top=10').")

    method, value_str = criteria.split("=", 1)
    method = method.strip().lower()
    value_str = value_str.strip()

    try:
        if method == "top":
            x = int(value_str)
            if x <= 0:
                raise ValueError("Число должно быть больше 0")
            filtered_data = sorted(data, key=lambda item: item['gdp'], reverse=True)[:x]
        elif method == "tail":
            x = int(value_str)
            if x <= 0:
                raise ValueError("Число должно быть больше 0")
            filtered_data = sorted(data, key=lambda item: item['gdp'])[:x]
        elif method == "greater":
            x = float(value_str)
            filtered_data = [item for item in sorted(data, key=lambda item: item['gdp'], reverse=True) if item['gdp'] > x]
        elif method == "less":
            x = float(value_str)
            filtered_data = [item for item in sorted(data, key=lambda item: item['gdp']) if item['gdp'] < x]
        else:
            raise IllegalArgumentError(f"Неизвестный метод '{method}'. Допустимые: top, tail, greater, less.")
    except ValueError as e:
        raise IllegalArgumentError(f"Неверное значение в критерии '{criteria}'. Ожидалось число. Детали: {e}")

    try:
        with open(filename, 'w', encoding='utf-8', newline='') as file:
            writer = csv.DictWriter(file, fieldnames=['name', 'gdp'])
            writer.writeheader()
            writer.writerows(filtered_data)
    except PermissionError:
        raise PermissionError(f"Нет прав на запись в файл '{filename}'.")
    except IOError as e:
        raise IOError(f"Ошибка ввода-вывода при записи в '{filename}': {e}")


class TestGDPProcessor(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.test_file = "test_wb_data.csv"
        cls.bad_headers_file = "test_bad_headers.csv"

        with open(cls.test_file, 'w', encoding='utf-8', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['name', 'gdp'])
            writer.writerow(['Luxembourg', '102831.32'])
            writer.writerow(['Switzerland', '78812.65'])
            writer.writerow(['Russian Federation', '8748.36'])
            writer.writerow(['American Samoa', ''])
            writer.writerow(['Country with text GDP', 'N/A'])
            writer.writerow(['', '5000.00'])

        with open(cls.bad_headers_file, 'w', encoding='utf-8', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['country', 'value'])

    @classmethod
    def tearDownClass(cls):
        for file in [cls.test_file, cls.bad_headers_file, "test_output.csv"]:
            if os.path.exists(file):
                os.remove(file)

    def test_load_data_success_and_skip(self):
        data = load_data(self.test_file)
        self.assertEqual(len(data), 3)
        self.assertEqual(data[0]['name'], 'Luxembourg')
        self.assertEqual(data[2]['name'], 'Russian Federation')

    def test_load_data_exceptions(self):
        with self.assertRaises(FileNotFoundError):
            load_data("non_existent_file.csv")
        with self.assertRaises(DataLoadError):
            load_data(self.bad_headers_file)

    def test_search_max_min_and_exact(self):
        data = load_data(self.test_file)
        self.assertEqual(search(data, "-max-")['name'], 'Luxembourg')
        self.assertEqual(search(data, "-min-")['name'], 'Russian Federation')
        self.assertEqual(search(data, "russian federation")['gdp'], 8748.36)

    def test_search_not_found(self):
        data = load_data(self.test_file)
        with self.assertRaises(NoSuchCountryError):
            search(data, "Atlantis")
        with self.assertRaises(NoSuchCountryError):
            search([], "-max-")

    def test_save_data_success(self):
        data = load_data(self.test_file)
        save_data("test_output.csv", data, "top=2")
        with open("test_output.csv", 'r', encoding='utf-8') as f:
            lines = f.readlines()
            self.assertEqual(len(lines), 3)
            self.assertIn("Luxembourg", lines[1])
            self.assertIn("Switzerland", lines[2])

    def test_save_data_exceptions(self):
        data = load_data(self.test_file)
        with self.assertRaises(IllegalArgumentError):
            save_data("test_output.csv", data, "unknown_method=5")
        with self.assertRaises(IllegalArgumentError):
            save_data("test_output.csv", data, "top=-5")
        with self.assertRaises(IllegalArgumentError):
            save_data("test_output.csv", data, "greater=abc")


if __name__ == "__main__":
    print("Запуск автоматических тестов...")
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(TestGDPProcessor)
    runner = unittest.TextTestRunner(verbosity=1)
    result = runner.run(suite)

    print("\n" + "=" * 50)
    if result.wasSuccessful():
        print("Все тесты пройдены успешно.")
    else:
        print("Обнаружены ошибки в тестах.")
    print("=" * 50 + "\n")

    print("Переход в интерактивный режим...")
    try:
        filename = input("Введите имя CSV-файла (или Enter для 'gdp_per_capita_2016 (1).csv'): ").strip()
        if not filename:
            filename = "gdp_per_capita_2016 (1).csv"

        save_filename = input("Введите имя файла для сохранения (или Enter для 'top_10_gdp.csv'): ").strip()
        if not save_filename:
            save_filename = "top_10_gdp.csv"

        print("\nЗагрузка данных...")
        data = load_data(filename)
        print(f"Загружено {len(data)} записей.")

        print("\nАнализ данных...")
        max_country = search(data, "-max-")
        print(f"Максимальный ВВП: {max_country['name']} ({max_country['gdp']:,.2f} $)")

        min_country = search(data, "-min-")
        print(f"Минимальный ВВП: {min_country['name']} ({min_country['gdp']:,.2f} $)")

        try:
            russia = search(data, "Russian Federation")
            print(f"Российская Федерация: {russia['gdp']:,.2f} $")
        except NoSuchCountryError:
            print("Российская Федерация не найдена в файле.")

        print("\nСохранение отфильтрованных данных...")
        save_data(save_filename, data, "top=10")
        print(f"Топ-10 стран сохранены в файл '{save_filename}'.")

    except FileNotFoundError as e:
        print(f"\nОшибка: Файл не найден. ({e})")
    except DataLoadError as e:
        print(f"\nОшибка: Некорректная структура файла. ({e})")
    except NoSuchCountryError as e:
        print(f"\nОшибка: {e}")
    except IllegalArgumentError as e:
        print(f"\nОшибка: Неверный формат критерия. ({e})")
    except PermissionError as e:
        print(f"\nОшибка: Нет прав доступа к файлу. ({e})")
    except Exception as e:
        print(f"\nНеожиданная ошибка: {e}")



......
----------------------------------------------------------------------
Ran 6 tests in 0.017s

OK


Saving gdp_per_capita_2016 (1).csv to gdp_per_capita_2016 (1).csv
Запуск автоматических тестов...

Все тесты пройдены успешно.

Переход в интерактивный режим...
Введите имя CSV-файла (или Enter для 'gdp_per_capita_2016 (1).csv'): 
Введите имя файла для сохранения (или Enter для 'top_10_gdp.csv'): 

Загрузка данных...
Загружено 184 записей.

Анализ данных...
Максимальный ВВП: Luxembourg (102,831.32 $)
Минимальный ВВП: Burundi (285.73 $)
Российская Федерация: 8,748.36 $

Сохранение отфильтрованных данных...
Топ-10 стран сохранены в файл 'top_10_gdp.csv'.
